[目录](./table_of_contents.ipynb)

# 无损卡尔曼滤波器

In [ ]:
%matplotlib inline

In [ ]:
# 格式化书
import book_format
book_format.set_style()

在上一章中，我们讨论了非线性系统所带来的困难。这种非线性可以出现在两个地方。它可以出现在我们的测量中，例如测量到物体的斜距。斜距需要你进行平方根运算来计算x，y坐标：

$$x=\sqrt{\text{slant}^2 - \text{altitude}^2}$$

非线性也可能出现在过程模型中 - 我们可能正在跟踪一个穿过空气的球，其中空气阻力的影响会导致非线性行为。标准卡尔曼滤波器在这些问题上表现不佳或根本无法工作。

在上一章中，我向您展示了这样的图表。我稍微改变了方程以强调非线性的影响。

In [ ]:
from kf_book.book_plots import set_figsize, figsize
import matplotlib.pyplot as plt
from kf_book.nonlinear_plots import plot_nonlinear_func
from numpy.random import normal
import numpy as np

# 创建具有均值为0、标准差为1的50万个样本
gaussian = (0., 1.)
data = normal(loc=gaussian[0], scale=gaussian[1], size=500000)

def f(x):
    return (np.cos(4*(x/2 + 0.7))) - 1.3*x

plot_nonlinear_func(data, f)

我是通过从输入中取500,000个样本，通过非线性变换并构建结果的直方图来生成这个的。我们把这些点称为 *sigma points*。从输出直方图中，我们可以计算出均值和标准偏差，从而得到一个更新的（虽然是近似的）高斯分布。

让我向您展示一下在通过 `f(x)` 之前和之后的数据的散点图。

In [ ]:
N = 30000
plt.subplot(121)
plt.scatter(data[:N], range(N), alpha=.2, s=1)
plt.title('输入')
plt.subplot(122)
plt.title('输出')
plt.scatter(f(data[:N]), range(N), alpha=.2, s=1);

这个数据本身看起来是高斯分布，确实是这样。我的意思是它看起来像是散布在零均值周围的白噪声。相比之下，`g(data)` 有一个明确定义的结构。有两个带状区域，中间有大量点。在带状区域的外面，有散布的点，但负数方向的点更多。

也许你已经意识到，这个采样过程构成了我们问题的解决方案。假设对于每次更新，我们生成 500,000 个点，将它们通过这个函数，然后计算结果的均值和方差。这被称为 *蒙特卡罗* 方法，它被一些 Kalman 滤波器设计所使用，例如集合滤波器和粒子滤波器。采样不需要专业知识，也不需要一个封闭形式的解。无论这个函数有多么非线性或者行为不良，只要我们用足够的 sigma 点进行采样，我们就能构建一个准确的输出分布。

"足够的点数"是关键。上面的图是用500,000个sigma点创建的，输出仍然不够平滑。更糟糕的是，这只是一个维度而已。所需点数随着维度数的增加而呈幂增长。如果在一个维度中只需要500个点，那么在两个维度中就需要500的平方，即250,000个点，在三个维度中需要500的立方，即125,000,000个点，依此类推。因此，虽然这种方法确实有效，但计算成本非常高。合奏滤波器和粒子滤波器使用巧妙的技术来显著减少这种维度，但计算负担仍然非常大。非线性卡尔曼滤波器使用sigma点，但通过使用确定性方法来选择点，大大减少了计算量。

## Sigma点-从分布中采样


让我们从二维协方差椭圆的角度来看待这个问题。我选择二维仅仅是因为它更容易绘制；但这可以扩展到任意维度。假设某个任意的非线性函数，我们将从第一个协方差椭圆中取随机点，通过非线性函数将它们传递，然后绘制它们的新位置。然后我们可以计算变换点的平均值和协方差，并将其用作我们对平均值和概率分布的估计。

In [ ]:
import kf_book.ukf_internal as ukf_internal
ukf_internal.show_2d_transform()

在左边，我们展示了一个椭圆，描绘了两个状态变量的$1\sigma$分布。箭头展示了几个随机抽样点如何通过某个任意的非线性函数转换到一个新的分布。右边的椭圆是半透明的，以表示它是这些点集的均值和方差的*估计*。

让我们编写一个函数，该函数通过非线性系统传递从高斯随机抽取的10000个点。

$$\mu = \begin{bmatrix}0\\0\end{bmatrix}, 
\Sigma=\begin{bmatrix}32&15\\15&40\end{bmatrix}$$

非线性系统如下所示：

$$\begin{cases}\begin{aligned}\bar x&=x+y\\
\bar y&= 0.1x^2 + y^2\end{aligned} \end{cases}$$

In [ ]:
import numpy as np
from numpy.random import multivariate_normal
from kf_book.nonlinear_plots import plot_monte_carlo_mean

def f_nonlinear_xy(x, y):
    return np.array([x + y, .1*x**2 + y*y])

mean = (0., 0.)
p = np.array([[32., 15.], [15., 40.]])
# 计算线性化的平均值
mean_fx = f_nonlinear_xy(*mean)

#生成随机点
xs, ys = multivariate_normal(mean=mean, cov=p, size=10000).T
plot_monte_carlo_mean(xs, ys, f_nonlinear_xy, mean_fx, '线性化平均值');

这个图展示了这个函数的强非线性性，以及如果我们按照扩展卡尔曼滤波器的方式进行线性化会导致的大误差（我们将在下一章学习这个）。

## 快速示例

我将很快进入无迹卡尔曼滤波器（UKF）用于选择 sigma 点并执行计算的数学部分。但让我们先从一个示例开始，这样你就可以看到目标，可以这么说。

我们将学习 UKF 可以使用多种不同的算法来生成 sigma 点。FilterPy 提供了几种算法。以下是一种可能性：

In [ ]:
from filterpy.kalman import JulierSigmaPoints

sigmas = JulierSigmaPoints(n=2, kappa=1)

这将在稍后变得更清晰，但该对象将为任何给定的均值和协方差生成加权 sigma 点。让我们看一个示例，其中点的大小表示它的加权程度：

In [ ]:
from kf_book.ukf_internal import plot_sigmas
plot_sigmas(sigmas, x=[3, 17], cov=[[1, .5], [.5, 3]])

可以看到，我们有5个点以奇怪的方式围绕着均值(3, 17)分布。这似乎很荒谬，但它的表现会比50万个随机生成的点好，甚至更好！

好的，现在让我们实现滤波器。我们将在一维中实现标准的线性滤波器；我们还没有准备好处理非线性滤波器。滤波器的设计与我们迄今所学习的内容没有太大区别，只有一个不同之处。KalmanFilter类使用矩阵$\mathbf F$来计算状态转移函数。矩阵意味着**线性**代数，适用于线性问题，但不适用于非线性问题。因此，我们提供一个函数，就像上面所做的那样，而不是一个矩阵。KalmanFilter类使用另一个矩阵$\mathbf H$来实现测量函数，将状态转换为等效的测量。同样，矩阵意味着线性，因此我们提供一个函数。也许很清楚为什么$\mathbf H$被称为“测量函数”了；对于线性卡尔曼滤波器，它是一个矩阵，但那只是一种快速计算线性函数的方法。

不再废话，这里是一个一维跟踪问题的状态转移函数和测量函数，其中状态为 $\mathbf x = [x \, \, \dot x]^ \mathsf T$：

In [ ]:
def fx(x, dt):
    xout = np.empty_like(x)
    xout[0] = x[1] * dt + x[0]
    xout[1] = x[1]
    return xout

def hx(x):
    return x[:1] # 返回位置 [x]

让我们明确一点，这是一个线性示例。对于线性问题，没有必要使用 UKF，但我从最简单的例子开始。但请注意，我编写了 `fx()` 来计算 $\mathbf{\bar x}$ 作为一组等式，而不是矩阵乘法。这是为了说明我可以在此实现任何任意的非线性函数；我们不受限于线性方程。

设计 $\mathbf P$、$\mathbf R$ 和 $\mathbf Q$ 的其余部分是相同的。你知道如何做到这一点，所以让我们完成滤波器并运行它。

In [ ]:
from numpy.random import randn
from filterpy.kalman import UnscentedKalmanFilter
from filterpy.common import Q_discrete_white_noise

ukf = UnscentedKalmanFilter(dim_x=2, dim_z=1, dt=1., hx=hx, fx=fx, points=sigmas)
ukf.P *= 10
ukf.R *= .5
ukf.Q = Q_discrete_white_noise(2, dt=1., var=0.03)

zs, xs = [], []
for i in range(50):
    z = i + randn()*.5
    ukf.predict()
    ukf.update(z)
    xs.append(ukf.x[0])
    zs.append(z)
    
plt.plot(xs);
plt.plot(zs, marker='x', ls='');

这里并没有什么新东西。您必须创建一个对象，为您创建 sigma 点，并提供函数而不是矩阵来表示 $\mathbf F$ 和 $\mathbf H$，但其余部分与以前相同。这应该足以让您对一些数学和算法有足够的信心，以便您可以理解 UKF 的运作原理。

## 选择 Sigma 点

在本章的开头，我使用了500,000个随机生成的sigma点来计算通过非线性函数的高斯分布的概率分布。虽然计算出的均值相当精确，但每次更新计算500,000个点会导致我们的滤波器变得极慢。那么，我们可以使用最少的采样点，这种问题的制定对点有什么样的约束？我们假设我们对非线性函数没有特殊的知识，因为我们想找到一个适用于任何函数的广义算法。

让我们考虑一下最简单的情况，看看它是否提供了一些见解。最简单的系统是**恒等函数**：$f(x) = x$。如果我们的算法不能处理恒等函数，那么滤波器就无法收敛。换句话说，如果输入为1（对于一维系统），输出也必须为1。如果输出不同，比如1.1，那么当我们在下一个时间步骤将1.1输入到变换中时，我们将得到另一个数字，可能是1.23。这个滤波器发散了。

我们可以使用每个维度一个点的最少数量。这就是线性Kalman滤波器使用的数量。对于分布$\mathcal{N}(\mu,\sigma^2)$的Kalman滤波器的输入是$\mu$本身。因此，虽然这对于线性情况有效，但对于非线性情况来说并不是一个好答案。

也许我们可以在每个维度上使用一个点，但需要进行一些修改。然而，如果我们将某个值 $\mu+\Delta$ 传递给恒等函数 $f(x)=x$，它将不会收敛，因此这种方法行不通。如果我们没有改变 $\mu$，那么这将是标准的卡尔曼滤波器。我们必须得出结论：一个样本将不起作用。

接下来我们可以选择的最小数量是多少呢？是两个。考虑高斯分布是对称的这一事实，以及我们可能希望始终将一个样本点选为输入均值以使恒等函数正常工作。两个样本点需要我们选择均值，然后再选择另一个点。那个点将会在输入中引入不对称性，这可能不是我们想要的。这将很难使恒等函数 $f(x)=x$ 正常工作。

接下来最小的数量是三个点。三个点允许我们选择均值，然后在均值的两侧各选择一个点，如下图所示。

In [ ]:
ukf_internal.show_3_sigma_points()

我们可以通过非线性函数f(x)传递这些点并计算结果的均值和方差。 均值可以计算为3个点的平均值，但这不是很通用。 例如，对于非常非线性的问题，我们可能希望将中心点的权重高得多，而不是外部点，或者我们可能希望将外部点的权重更高。

一个更通用的方法是计算加权平均值$\mu = \sum_i w_i\, f(\mathcal{X}_i)$，其中花体字母$\mathcal{X}$是sigma点。 我们需要权重的总和为1。 鉴于这个要求，我们的任务是选择$\mathcal{X}$及其相应的权重，以便它们计算为转换后的sigma点的均值和方差。

如果加权平均值，加权协方差也是有意义的。 可以为均值($w^m$)和协方差($w^c$)使用不同的权重。 我们使用上标来允许在以下方程中加索引。 我们可以写成：

$$\begin{aligned}
\mathbf{约束条件:}\\
1 &= \sum_i{w_i^m} \\
1 &= \sum_i{w_i^c} \\
\mu &= \sum_i w_i^mf(\mathcal{X}_i) \\
\Sigma &= \sum_i w_i^c{(f(\mathcal{X})_i-\mu)(f(\mathcal{X})_i-\mu)^\mathsf{T}}
\end{aligned}
$$

前两个方程是权重必须加起来为一的约束条件。第三个方程是计算加权平均值的方法。第四个方程可能不太熟悉，但是请记住两个随机变量协方差的方程如下：

$$COV(x,y) = \frac{\sum(x-\bar x)(y-\bar{y})}{n}$$

这些约束条件并不形成唯一的解。例如，如果使 $w^m_0$ 更小，则可以通过使 $w^m_1$ 和 $w^m_2$ 更大来进行补偿。您可以为均值和协方差使用不同的权重，也可以使用相同的权重。事实上，这些方程并不要求任何点都是输入均值，尽管这似乎“好”这么做。

我们希望有一个算法能够满足约束条件，最好每个维度只有3个点。在继续之前，我想确保这个想法是清楚的。下面有三个不同的示例，用于相同协方差椭圆，但使用了不同的sigma点。每个sigma点的大小与给定的权重成比例。

In [ ]:
ukf_internal.show_sigma_selections()

这些点不沿着椭圆的主轴和次轴排列；约束条件中没有要求我这样做。我将这些点均匀分布，但约束条件并不要求这样做。

sigma points的布置和加权方式会影响我们对分布的采样。彼此靠近的点将采样局部效应，因此对于非常非线性的问题可能效果更好。距离较远或远离椭圆轴线的点将采样非局部效应和非高斯行为。然而，通过改变每个点使用的权重，我们可以减轻这种影响。如果点距离平均值很远但权重非常小，我们将纳入关于分布的一些知识，而不允许问题的非线性性产生错误的估计。

请理解，选择sigma points的方法是无限的。我选择的约束只是其中一种方法。例如，并非所有创建sigma points的算法都要求权重总和为一。实际上，我在本书中青睐的算法就没有这个属性。

## 无香精变换

暂时假设存在一种选择sigma点和权重的算法。如何使用sigma点来实现滤波器？

*无损变换*是该算法的核心，但它非常简单。它通过一个非线性函数传递sigma点$\boldsymbol{\chi}$，得到一组变换后的点。

$$\boldsymbol{\mathcal{Y}} = f(\boldsymbol{\chi})$$

然后计算变换后点的均值和协方差。该均值和协方差成为新的估计值。下图描述了无损变换的操作。右侧的绿色椭圆表示计算的均值和协方差，用于变换后的sigma点。

In [ ]:
ukf_internal.show_sigma_transform(with_text=True)

计算sigma点的均值和协方差：

$$\begin{aligned}
\mu &= \sum_{i=0}^{2n} w^m_i\boldsymbol{\mathcal{Y}}_i \\
\Sigma &= \sum_{i=0}^{2n} w^c_i{(\boldsymbol{\mathcal{Y}}_i-\mu)(\boldsymbol{\mathcal{Y}}_i-\mu)^\mathsf{T}}
\end{aligned}
$$

这些方程应该很熟悉 - 它们是我们上面开发的约束方程。

简而言之，无迹变换采取从任意概率分布中采样的点，通过任意非线性函数将它们传递，并为每个变换点产生一个高斯分布。我希望您能设想我们如何使用它来实现非线性卡尔曼滤波器。一旦我们有了高斯分布，我们已经开发的所有数学工具就会发挥作用！

“无迹”这个名字可能会让人困惑。它实际上并没有什么意义。这是由发明者开玩笑取的名字，他的算法并没有“臭味”，很快这个名字就流行了起来。这个术语没有数学意义。

### 无迹变换的准确性

之前我们编写了一个函数，通过将 50,000 个点通过非线性函数来找到分布的平均值。现在让我们通过相同的函数将 5 个 sigma 点传递，然后使用 unscented transform 计算它们的平均值。我们将使用 FilterPy 函数 *MerweScaledSigmaPoints()* 来创建 sigma 点，使用 `unscented_transform` 来执行变换；我们稍后会介绍这些函数。在本章的第一个例子中，我使用了 `JulierSigmaPoints`；它们都选择 sigma 点，但是以不同的方式选择，我稍后会解释。

In [ ]:
from filterpy.kalman import unscented_transform, MerweScaledSigmaPoints
import scipy.stats as stats

# 初始均值和协方差
mean = (0., 0.)
p = np.array([[32., 15], [15., 40.]])

# 创建 sigma 点和权重
points = MerweScaledSigmaPoints(n=2, alpha=.3, beta=2., kappa=.1)
sigmas = points.sigma_points(mean, p)

### 通过非线性函数
sigmas_f = np.empty((5, 2))
for i in range(5):
    sigmas_f[i] = f_nonlinear_xy(sigmas[i, 0], sigmas[i ,1])

### 使用无损变换获取新的均值和协方差
ukf_mean, ukf_cov = unscented_transform(sigmas_f, points.Wm, points.Wc)

# 生成随机点
np.random.seed(100)
xs, ys = multivariate_normal(mean=mean, cov=p, size=5000).T

plot_monte_carlo_mean(xs, ys, f_nonlinear_xy, ukf_mean, '无损均值')
ax = plt.gcf().axes[0]
ax.scatter(sigmas[:,0], sigmas[:,1], c='r', s=30);

我认为这个结果非常显著。仅使用5个点，我们就能以惊人的精度计算出平均值。x方向的误差仅为-0.097，y方向的误差为0.549。相比之下，线性化方法（由下一章节中将要学习的EKF使用）在y方向上的误差超过了43。如果你看一下生成sigma点的代码，你会发现它并不知道非线性函数的存在，只知道我们初始分布的均值和协方差。如果我们有完全不同的非线性函数，同样的5个sigma点将被生成。

我承认选择了一个非线性函数，使得无损变换的性能与EKF相比更为显著。但物理世界充满了非常非线性的行为，而UKF则可以轻松地处理。我并没有“努力”寻找一个无损变换恰好运行良好的函数。您将在下一章中看到更传统的技术如何应对强烈的非线性。这张图是为什么我建议您尽可能使用UKF或类似现代技术的基础。

## 无损卡尔曼滤波器

现在我们可以介绍UKF算法。

### 预测步骤

UKF的预测步骤使用过程模型 $f()$ 计算先验。$f()$ 被假定为非线性，因此我们根据某个函数生成sigma点 $\mathcal{X}$ 及其对应的权重 $W^m, W^c$：

$$\begin{aligned}
\boldsymbol\chi &= \text{sigma-function}(\mathbf x, \mathbf P) \\
W^m, W^c &= \text{weight-function}(\mathtt{n, parameters})\end{aligned}$$

我们将每个sigma点通过$f(\mathbf x, \Delta t)$。这将根据过程模型将sigma点向前推进时间，形成新的先验，这是我们命名的一组sigma点$\boldsymbol{\mathcal Y}$：

$$\boldsymbol{\mathcal{Y}} = f(\boldsymbol{\chi}, \Delta t)$$

我们使用变换后的sigma点上的*无迹变换*来计算先验的均值和协方差。

$$\mathbf{\bar x}, \mathbf{\bar P} = 
UT(\mathcal{Y}, w_m, w_c, \mathbf Q)$$

这些是无迹变换的方程：

$$\begin{aligned}
\mathbf{\bar x} &= \sum_{i=0}^{2n} w^m_i\boldsymbol{\mathcal Y}_i \\
\mathbf{\bar P} &= \sum_{i=0}^{2n} w^c_i({\boldsymbol{\mathcal Y}_i - \mathbf{\bar x})(\boldsymbol{\mathcal Y}_i-\mathbf{\bar x})^\mathsf{T}} + \mathbf Q
\end{aligned}
$$

此表比较了线性Kalman滤波器和无迹Kalman滤波器方程。出于可读性，我省略了下标$i$。

$$\begin{array}{l|l}
\text{卡尔曼} & \text{无迹} \\
\hline 
& \boldsymbol{\mathcal Y} = f(\boldsymbol\chi) \\
\mathbf{\bar x} = \mathbf{Fx} & 
\mathbf{\bar x} = \sum w^m\boldsymbol{\mathcal Y}  \\
\mathbf{\bar P} = \mathbf{FPF}^\mathsf T + \mathbf Q  & 
\mathbf{\bar P} = \sum w^c({\boldsymbol{\mathcal Y} - \mathbf{\bar x})(\boldsymbol{\mathcal Y} - \mathbf{\bar x})^\mathsf T}+\mathbf Q
\end{array}$$

### 更新步骤

卡尔曼滤波器将更新步骤执行在测量空间中。因此，我们必须使用您定义的测量函数 $h(x)$ 将先前的 sigma 点转换为测量值。

$$\boldsymbol{\mathcal{Z}} = h(\boldsymbol{\mathcal{Y}})$$

我们使用无迹变换计算这些点的均值和协方差。$z$ 下标表示这些是测量 sigma 点的均值和协方差。

$$\begin{aligned}
\boldsymbol\mu_z, \mathbf P_z &= 
UT(\boldsymbol{\mathcal Z}, w_m, w_c, \mathbf R) \\
\boldsymbol\mu_z &= \sum_{i=0}^{2n} w^m_i\boldsymbol{\mathcal Z}_i \\
\mathbf P_z &= \sum_{i=0}^{2n} w^c_i{(\boldsymbol{\mathcal Z}_i-\boldsymbol{\mu}_z)(\boldsymbol{\mathcal Z}_i-\boldsymbol{\mu}_z)^\mathsf T} + \mathbf R
\end{aligned}
$$

然后我们计算残差和卡尔曼增益。测量 $\mathbf z$ 的残差很容易计算：

$$\mathbf y = \mathbf z - \boldsymbol\mu_z$$

为了计算卡尔曼增益，我们首先计算状态和测量的[交叉协方差](https://en.wikipedia.org/wiki/Cross-covariance)，它被定义为：

$$\mathbf P_{xz} =\sum_{i=0}^{2n} w^c_i(\boldsymbol{\mathcal Y}_i-\mathbf{\bar x})(\boldsymbol{\mathcal Z}_i-\boldsymbol\mu_z)^\mathsf T$$

然后卡尔曼增益被定义为：

$$\mathbf{K} = \mathbf P_{xz} \mathbf P_z^{-1}$$

如果将矩阵的倒数看作是一种矩阵倒置，你可以看到卡尔曼增益是一个简单的比率，用于计算：

$$\mathbf{K} \approx \frac{\mathbf P_{xz}}{\mathbf P_z} 
\approx \frac{\text{对状态的置信}}{\text{对测量的置信}}$$

最后，我们使用残差和卡尔曼增益计算新的状态估计：

$$\mathbf x = \bar{\mathbf x} + \mathbf{Ky}$$

新协方差的计算方式为：

$$ \mathbf P = \mathbf{\bar P} - \mathbf{KP_z}\mathbf{K}^\mathsf{T}$$

这一步包含了一些需要信任的方程，但你应该能够看到它们与线性卡尔曼滤波方程的关系。线性代数与线性卡尔曼滤波略有不同，但算法是本书中一直实现的相同贝叶斯算法。 

这张表比较了线性KF和UKF方程。

$$\begin{array}{l|l}
\textrm{卡尔曼滤波器} & \textrm{无迹卡尔曼滤波器} \\
\hline 
& \boldsymbol{\mathcal Y} = f(\boldsymbol\chi) \\
\mathbf{\bar x} = \mathbf{Fx} & 
\mathbf{\bar x} = \sum w^m\boldsymbol{\mathcal Y}  \\
\mathbf{\bar P} = \mathbf{FPF}^\mathsf T+\mathbf Q  & 
\mathbf{\bar P} = \sum w^c({\boldsymbol{\mathcal Y} - \mathbf{\bar x})(\boldsymbol{\mathcal Y} - \mathbf{\bar x})^\mathsf T}+\mathbf Q \\
\hline 
& \boldsymbol{\mathcal Z} =  h(\boldsymbol{\mathcal{Y}}) \\
& \boldsymbol\mu_z = \sum w^m\boldsymbol{\mathcal{Z}} \\
\mathbf y = \mathbf z - \mathbf{Hx} &
\mathbf y = \mathbf z - \boldsymbol\mu_z \\
\mathbf S = \mathbf{H\bar PH}^\mathsf{T} + \mathbf R & 
\mathbf P_z = \sum w^c{(\boldsymbol{\mathcal Z}-\boldsymbol\mu_z)(\boldsymbol{\mathcal{Z}}-\boldsymbol\mu_z)^\mathsf{T}} + \mathbf R \\ 
\mathbf K = \mathbf{\bar PH}^\mathsf T \mathbf S^{-1} &
<End of Text>

In [ ]:
$\mathbf K = \left[\sum w^c(\boldsymbol{\mathcal Y}-\bar{\mathbf x})(\boldsymbol{\mathcal{Z}}-\boldsymbol\mu_z)^\mathsf{T}\right] \mathbf P_z^{-1}$

$\mathbf x = \mathbf{\bar x} + \mathbf{Ky}$ & $\mathbf x = \mathbf{\bar x} + \mathbf{Ky}$

$\mathbf P = (\mathbf{I}-\mathbf{KH})\mathbf{\bar P}$ & $\mathbf P = \bar{\mathbf P} - \mathbf{KP_z}\mathbf{K}^\mathsf{T}$

## Van der Merwe的缩放Sigma点算法

有许多算法可用于选择Sigma点。自2005年左右以来，研究和工业界大多采用Rudolph Van der Merwe在他2004年的博士论文[1]中发表的版本。它在各种问题上表现良好，并在性能和精度之间有良好的平衡。这是对由Simon J. Julier [2]发表的*缩放无味变换*的轻微重组。

该公式使用3个参数来控制Sigma点的分布和加权：$\alpha$，$\beta$和$\kappa$。在我们处理方程式之前，让我们来看一个例子。我将在协方差椭圆形上绘制Sigma点，显示第一和第二标准差，并根据平均权重缩放点。

In [ ]:
ukf_internal.plot_sigma_points()

我们可以看到，sigma点位于第一和第二个标准差之间，较大的$\alpha$会扩散这些点。此外，较大的$\alpha$会比较小的$\alpha$更加权重平均值(中心点)，并更少地对其他点进行加权。这应符合我们的直觉-点距离平均值越远，我们就应该更少地给它分配权重。我们还不知道如何选择这些权重和sigma点，但选择看起来是合理的。

### Sigma点计算

第一个sigma点是输入的平均值。这是图中椭圆中心处显示的sigma点。我们将其称为$\boldsymbol{\chi}_0$。

$$ \mathcal{X}_0 = \mu$$

为了方便表示，我们定义$\lambda = \alpha^2(n+\kappa)-n$，其中$n$是$\mathbf x$的维数。其余的sigma点计算如下：

$$ 
\boldsymbol{\chi}_i = \begin{cases}
\mu + \left[ \sqrt{(n+\lambda)\Sigma}\right ]_{i}& i=1..n \\
\mu - \left[ \sqrt{(n+\lambda)\Sigma}\right]_{i-n} &i=(n+1)..2n\end{cases}
$$
其中，$i$ 下标选择了矩阵的第 $i$ 行向量。

换句话说，我们通过一个常数来缩放协方差矩阵，对其进行平方根处理，并通过从均值中加减协方差矩阵来确保对称性。我们将在稍后讨论如何对矩阵进行平方根处理。

### 权重计算

这个公式使用了一组权重来计算均值和另一组权重来计算协方差。$\mathcal{X}_0$ 的均值权重计算公式为：

$$W^m_0 = \frac{\lambda}{n+\lambda}$$

$\mathcal{X}_0$ 的协方差权重为：

$$W^c_0 = \frac{\lambda}{n+\lambda} + 1 -\alpha^2 + \beta$$

其余的 sigma 点 $\boldsymbol{\chi}_1 ... \boldsymbol{\chi}_{2n}$ 的均值和协方差的权重相同。它们是：

$$W^m_i = W^c_i = \frac{1}{2(n+\lambda)}\;\;\;i=1..2n$$

这为什么是“正确的”可能不是很明显，实际上，不能证明这对所有非线性问题都是理想的。但是，您可以看到我们选择的sigma点与协方差矩阵的平方根成比例，方差的平方根是标准差。因此，sigma点根据某些缩放因子大致按照$\pm 1\sigma$进行扩展。分母中有一个$n$项，因此随着维数的增加，点将被展开并且权重将减小。

**重要提示：**通常，这些权重不会求和为1。我收到很多关于这个问题的问题。预计将权重求和大于1甚至是负值。我在下面更详细地介绍了这一点。


### 参数的合理选择

当处理高斯问题时，$\beta=2$ 是一个好的选择，当 $n$ 是 $\mathbf x$ 的维度时，$\kappa=3-n$ 是一个好的选择，对于 $\alpha$，$0 \le \alpha \le 1$ 是一个合适的选择，其中较大的 $\alpha$ 值会将 sigma 点从均值分散得更远。

## 使用 UKF

让我们解决一些问题，以便您可以获得对使用 UKF 的便利性的信心。我们将先从一个您已经知道如何使用线性卡尔曼滤波器解决的线性问题开始。虽然 UKF 是为非线性问题设计的，但对于线性问题，它找到的最优结果与线性卡尔曼滤波器相同。我们将编写一个滤波器来使用恒定速度模型跟踪二维中的一个对象。这将使我们关注相同的内容（大部分相同！）以及 UKF 中有何不同。

设计卡尔曼滤波器需要您指定$\bf{x}$、$\bf{F}$、$\bf{H}$、$\bf{R}$和$\bf{Q}$矩阵。我们已经做了很多次，所以我会给你这些矩阵，不做太多讨论。我们需要一个恒定速度模型，因此我们将$\bf{x}$定义为

$$ \mathbf x = \begin{bmatrix}x &  \dot x & y & \dot y \end{bmatrix}^\mathsf{T}$$

使用这种状态变量的排序方式，状态转移矩阵为

$$\mathbf F = \begin{bmatrix}1 & \Delta t & 0 & 0 \\
0&1&0&0 \\
0&0&1&\Delta t\\
0&0&0&1
\end{bmatrix}$$

它实现了牛顿方程

$$\begin{aligned}
x_k &= x_{k-1} + \dot x_{k-1}\Delta t \\
y_k &= y_{k-1} + \dot y_{k-1}\Delta t
\end{aligned}$$

我们的传感器提供位置信息，但不提供速度信息，因此测量函数为

$$\mathbf H = \begin{bmatrix}1&0&0&0 \\ 0&0&1&0
\end{bmatrix}$$

传感器读数以米为单位，*x*和*y*的误差均为$\sigma=0.3$米。这给我们带来了一个测量噪声矩阵。

$$\mathbf R = \begin{bmatrix}0.3^2 &0\\0 & 0.3^2\end{bmatrix}$$

最后，我们假设过程噪声可以用离散白噪声模型来表示——也就是说，在每个时间段内加速度是恒定的。我们可以使用 `FilterPy` 的 `Q_discrete_white_noise()` 来创建这个矩阵，但为了审查，该矩阵为：

$$\mathbf Q = \begin{bmatrix}
\frac{1}{4}\Delta t^4 & \frac{1}{2}\Delta t^3 \\
\frac{1}{2}\Delta t^3 & \Delta t^2\end{bmatrix} \sigma^2$$

我对这个滤波器的实现是：

In [ ]:
from filterpy.kalman import KalmanFilter
from filterpy.common import Q_discrete_white_noise
from numpy.random import randn

std_x，std_y = .3，.3
dt = 1.0

In [ ]:
np.random.seed(1234)
kf = KalmanFilter(4, 2)
kf.x = np.array([0., 0., 0., 0.])
kf.R = np.diag([std_x**2, std_y**2])
kf.F = np.array([[1, dt, 0, 0], 
                 [0, 1, 0, 0],
                 [0, 0, 1, dt],
                 [0, 0, 0, 1]])
kf.H = np.array([[1, 0, 0, 0],
                 [0, 0, 1, 0]])
 
kf.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=1, var=0.02)
kf.Q[2:4, 2:4] = Q_discrete_white_noise(2, dt=1, var=0.02)

zs = [np.array([i + randn()*std_x, 
                i + randn()*std_y]) for i in range(100)]               
xs, _, _, _ = kf.batch_filter(zs)
plt.plot(xs[:, 0], xs[:, 2]);

这应该对你来说没有什么意外。现在让我们实现一个UKF。同样，这纯粹是为了教育目的而实现的；在线性问题上使用UKF没有任何好处。`FilterPy`使用类`UnscentedKalmanFilter`实现了UKF。

第一件事是实现函数`f(x, dt)`和`h(x)`。`f(x, dt)`实现状态转移函数，而`h(x)`实现测量函数。这些对应于线性滤波器中的矩阵$\mathbf F$和$\mathbf H$。

下面是这两个函数的一个合理实现。每个函数都应返回包含结果的1D NumPy数组或列表。你可以给它们比`f`和`h`更易读的名称。

In [ ]:
def f_cv(x, dt):
    """一个常速飞机的状态转移函数"""
    
    F = np.array([[1, dt, 0,  0],
                  [0,  1, 0,  0],
                  [0,  0, 1, dt],
                  [0,  0, 0,  1]])
    return F @ x

def h_cv(x):
    return x[[0, 2]]

接下来，您需要指定如何计算sigma点和权重。我们上面提供了Van der Merwe的版本，但有许多不同的选择。FilterPy使用一个名为`SigmaPoints`的类，该类必须实现一个方法：

In [ ]:
def sigma_points(self, x, P)

并包含属性 `Wm` 和 `Wc`，分别保存用于计算均值和协方差的权重。

FilterPy从 `SigmaPoints` 派生类 `MerweScaledSigmaPoints` 并实现了上述方法。

当您创建UKF时，将传入 $f()$ 和 $h()$ 函数以及sigma点对象，如以下示例：

In [ ]:
from filterpy.kalman import MerweScaledSigmaPoints
from filterpy.kalman import UnscentedKalmanFilter as UKF

points = MerweScaledSigmaPoints(n=4, alpha=.1, beta=2., kappa=-1)
ukf = UKF(dim_x=4, dim_z=2, fx=f_cv, hx=h_cv, dt=dt, points=points)

代码的其余部分与线性卡尔曼滤波器相同。我将使用相同的测量值并计算两个解之间差异的标准偏差。

In [ ]:
from filterpy.kalman import UnscentedKalmanFilter as UKF

import numpy as np

In [ ]:
sigmas = MerweScaledSigmaPoints(4, alpha=.1, beta=2., kappa=1.)
ukf = UKF(dim_x=4, dim_z=2, fx=f_cv,
          hx=h_cv, dt=dt, points=sigmas)
ukf.x = np.array([0., 0., 0., 0.])
ukf.R = np.diag([0.09, 0.09]) 
ukf.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=1, var=0.02)
ukf.Q[2:4, 2:4] = Q_discrete_white_noise(2, dt=1, var=0.02)

uxs = []
for z in zs:
    ukf.predict()
    ukf.update(z)
    uxs.append(ukf.x.copy())
uxs = np.array(uxs)

plt.plot(uxs[:, 0], uxs[:, 2])
print(f'UKF标准差为{np.std(uxs - xs):.3f}米')

这给出了标准差为0.013米的结果，非常小。

UKF的实现与线性Kalman滤波器并没有太大的区别。你提供非线性函数`f()`和`h()`，而不是实现状态转移和测量函数作为矩阵$\mathbf F$和$\mathbf H$。理论和实现的其余部分保持不变。实现`predict()`和`update()`的代码会有所不同，但从设计者的角度来看，问题的制定和滤波器设计非常相似。

## 跟踪飞机

让我们来解决我们的第一个非线性问题。我们将编写一个滤波器，使用雷达作为传感器来跟踪飞机。为了让问题尽可能与之前的问题相似，我们将在两个维度上进行跟踪。我们将在地面上跟踪一个维度和飞机的高度。每个维度是独立的，因此我们可以毫不损失地这样做。

雷达通过发射射频或微波来工作。光束路径上的任何物体都会将一部分信号反射回雷达。通过计算反射信号返回的时间，可以计算出目标的斜距。斜距是从雷达到目标物体的直线距离。方位角是使用天线的指向增益来计算的。

我们通过斜距和仰角来计算飞机的(x,y)位置，如下图所示：

In [ ]:
import kf_book.ekf_internal as ekf_internal
ekf_internal.show_radar_chart()

*仰角* $\epsilon$ 是与地面形成的视线的角度。

我们将假设飞机以恒定的高度飞行。因此，我们有一个三变量状态向量：

$$\mathbf x = \begin{bmatrix}\mathtt{距离} \\\mathtt{速度}\\ \mathtt{高度}\end{bmatrix}=    \begin{bmatrix}x \\ \dot x\\ y\end{bmatrix}$$

状态转移函数是线性的。

$$\mathbf{\bar x} = \begin{bmatrix} 1 & \Delta t & 0 \\ 0& 1& 0 \\ 0&0&1\end{bmatrix}
\begin{bmatrix}x \\ \dot x\\ y\end{bmatrix}
$$

并且可以使用以下代码计算：

In [ ]:
def f_radar(x, dt):
    """常速飞机状态转移函数，状态向量为[x, 速度, 高度]'"""
    
    F = np.array([[1, dt, 0],
                  [0,  1, 0],
                  [0,  0, 1]], dtype=float)
    return F @ x

接下来我们设计测量函数。与线性卡尔曼滤波器类似，测量函数将滤波器的先验转换为测量值。我们需要将飞机的位置和速度转换为从雷达站的仰角和距离。

距离通过勾股定理计算：

$$\text{距离} = \sqrt{(x_\text{飞机} - x_\text{雷达})^2 + (y_\text{飞机} - y_\text{雷达})^2}$$

仰角 $\epsilon$ 是 $y/x$ 的反正切：

$$\epsilon = \tan^{-1}{\frac{y_\mathtt{ac} - y_\text{radar}}{x_\text{ac} - x_\text{radar}}}$$

我们需要定义一个Python函数来计算它。我将利用一个函数可以拥有一个变量来存储雷达的位置的事实。虽然这对于这个问题来说不是必要的（我们可以硬编码该值，或者使用一个全局变量），但这使得函数更加灵活。

In [ ]:
def h_radar(x):
    dx = x[0] - h_radar.radar_pos[0]
    dy = x[2] - h_radar.radar_pos[1]
    slant_range = math.sqrt(dx**2 + dy**2)
    elevation_angle = math.atan2(dy, dx)
    return [slant_range, elevation_angle]

h_radar.radar_pos = (0, 0)

有一个非线性因素我们没有考虑到，那就是角度是模块化的。残差是测量值和先前投影到测量空间的差异。359°和1°之间的角度差是2°，但是359° - 1° = 358°。由于UKF计算未膨胀变换中加权值的总和，这种情况更加恶化。现在我们将把传感器和目标放置在避免这些非线性区域的位置上。稍后我会向你展示如何处理这个问题。

我们需要模拟雷达和飞机。现在你应该已经非常熟悉了，所以我直接提供代码而不进行讨论。

In [ ]:
from numpy.linalg import norm
from math import atan2

class RadarStation:
    
    def __init__(self, pos, range_std, elev_angle_std):
        self.pos = np.asarray(pos)       
        self.range_std = range_std
        self.elev_angle_std = elev_angle_std

In [ ]:
def reading_of(self, ac_pos):
        """返回到飞机的（距离，仰角）。仰角以弧度为单位。
        """
        
        diff = np.subtract(ac_pos, self.pos)
        rng = norm(diff)
        brg = atan2(diff[1], diff[0])
        return rng, brg


    def noisy_reading(self, ac_pos):
        """计算到飞机的距离和仰角，带有模拟noise"""
        
        rng, brg = self.reading_of(ac_pos)      
        rng += randn() * self.range_std
        brg += randn() * self.elev_angle_std 
        return rng, brg

In [ ]:
class ACSim:
    def __init__(self, pos, vel, vel_std):
        self.pos = np.asarray(pos, dtype=float)
        self.vel = np.asarray(vel, dtype=float)
        self.vel_std = vel_std        
        
    def update(self, dt):
        """计算并返回下一个位置。包含速度随机变化。"""
        
        dx = self.vel*dt + (randn() * self.vel_std) * dt      
        self.pos += dx     
        return self.pos

军用雷达可以达到1米的距离RMS精度和1毫弧度的高程角RMS精度[3]。我们假设更适中的精度为5米的距离精度和0.5度的角度精度，因为这提供了一个更具挑战性的数据集用于滤波器。

$\mathbf Q$ 的设计需要一些讨论。状态为 $\begin{bmatrix}x & \dot x & y\end{bmatrix}^\mathtt{T}$。前两个元素是下行距离和速度，因此我们可以使用 `Q_discrete_white_noise` 噪声来计算 Q 的左上角的值。第三个元素是高度，我们假设它与 $x$ 独立。这导致了 $\mathbf Q$ 的块设计：

$$\mathbf Q = \begin{bmatrix}\mathbf Q_\mathtt{x} & \boldsymbol 0 \\ \boldsymbol 0 & Q_\mathtt{y}\end{bmatrix}$$

我将从飞机直接位于雷达站上方，以100 m/s的速度飞行开始。一个典型的高度查找雷达可能只更新一次，每3秒钟更新一次，因此我们将使用这个作为我们的时代周期。

In [ ]:
import math
from kf_book.ukf_internal import plot_radar

dt = 3. # 读数之间间隔12秒
range_std = 5 # 米
elevation_angle_std = math.radians(0.5)
ac_pos = (0., 1000.)
ac_vel = (100., 0.)
radar_pos = (0., 0.)
h_radar.radar_pos = radar_pos

points = MerweScaledSigmaPoints(n=3, alpha=.1, beta=2., kappa=0.)
kf = UKF(3, 2, dt, fx=f_radar, hx=h_radar, points=points)

kf.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=dt, var=0.1)
kf.Q[2,2] = 0.1

kf.R = np.diag([range_std**2, elevation_angle_std**2])
kf.x = np.array([0., 90., 1100.])
kf.P = np.diag([300**2, 30**2, 150**2])

np.random.seed(200)
pos = (0, 0)
radar = RadarStation(pos, range_std, elevation_angle_std)
ac = ACSim(ac_pos, (100, 0), 0.02)

time = np.arange(0, 360 + dt, dt)
xs = []
for _ in time:
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    kf.predict()
    kf.update([r[0], r[1]]) 
    xs.append(kf.x)    
plot_radar(xs, time)

这可能会或可能不会给你留下深刻的印象，但对我来说却给我留下了深刻的印象！在扩展卡尔曼滤波器章节中，我们将解决相同的问题，但需要使用大量的数学。 

### 追踪机动飞行器

前面的例子效果不错，但它假设飞行器不改变高度。如果飞机在一分钟后开始爬升，以下是滤波器的结果。

In [ ]:
from kf_book.ukf_internal import plot_altitude

# 重置飞机位置
kf.x = np.array([0., 90., 1100.])
kf.P = np.diag([300**2, 30**2, 150**2])
ac = ACSim(ac_pos, (100, 0), 0.02)

np.random.seed(200)
time = np.arange(0, 360 + dt, dt)
xs, ys = [], []
for t in time:
    if t >= 60:
        ac.vel[1] = 300/60 # 300米/分钟的爬升率
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    ys.append(ac.pos[1])
    kf.predict()
    kf.update([r[0], r[1]]) 
    xs.append(kf.x)

plot_altitude(xs, time, ys)
print(f'实际高度：{ac.pos[1]:.1f}')
print(f'UKF高度：{xs[-1][2]:.1f}')

滤波器无法跟踪高度的变化。我们需要在设计中做出什么改变呢？

希望你的答案是“将爬升率添加到状态中”，如下所示：

$$\mathbf x = \begin{bmatrix}\mathtt{距离} \\\mathtt{速度}\\ \mathtt{高度} \\ \mathtt{爬升率}\end{bmatrix}=  \begin{bmatrix}x \\\dot x\\ y \\ \dot y\end{bmatrix}$$

这需要对状态转移函数进行以下更改，但仍然是线性的。

$$\mathbf{F} = \begin{bmatrix} 1 & \Delta t & 0 &0 \\ 0& 1& 0 &0\\ 0&0&1&\Delta t \\ 0&0&0&1\end{bmatrix}
\begin{bmatrix}x \\\dot x\\ y\\ \dot y\end{bmatrix} 
$$

测量函数保持不变，但我们必须更改 $\mathbf Q$ 来解决 $\mathbf x$ 维度变化的问题。

In [ ]:
def f_cv_radar(x, dt):
    """这是一个用于常速飞机的状态转移函数"""
    F = np.array([[1, dt, 0, 0],
                  [0,  1, 0, 0],
                  [0,  0, 1, dt],
                  [0,  0, 0, 1]], dtype=float)
    return F @ x
    
def cv_UKF(fx, hx, R_std):
    points = MerweScaledSigmaPoints(n=4, alpha=.1, beta=2., kappa=-1.)
    kf = UKF(4, len(R_std), dt, fx=fx, hx=hx, points=points)

    kf.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=dt, var=0.1)
    kf.Q[2:4, 2:4] = Q_discrete_white_noise(2, dt=dt, var=0.1)
    kf.R = np.diag(R_std)
    kf.R = kf.R @ kf.R  # 用平方获得方差
    kf.x = np.array([0., 90., 1100., 0.])
    kf.P = np.diag([300**2, 3**2, 150**2, 3**2])
    return kf

In [ ]:
np.random.seed(200)
ac = ACSim(ac_pos, (100, 0), 0.02)

In [ ]:
kf_cv = cv_UKF(f_cv_radar, h_radar, R_std=[range_std, elevation_angle_std])
time = np.arange(0, 360 + dt, dt)
xs, ys = [], []
for t in time:
    if t >= 60:
        ac.vel[1] = 300/60 # 300 meters/minute climb
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    ys.append(ac.pos[1])
    kf_cv.predict()
    kf_cv.update([r[0], r[1]]) 
    xs.append(kf_cv.x)

plot_altitude(xs, time, ys)
print(f'实际高度: {ac.pos[1]:.1f}')
print(f'UKF高度   : {xs[-1][2]:.1f}')

虽然高度估计中引入了大量噪声，但我们现在可以准确地跟踪高度。

### 传感器融合

现在让我们考虑一个传感器融合的例子。我们有某种类型的多普勒系统，它以2 m/s的RMS精度产生速度估计。我说“某种类型”是因为与雷达一样，我不是在教你如何为多普勒系统创建精确的滤波器。完整的实现必须考虑信噪比、大气影响、系统的几何形状等因素。

在前面的例子中，雷达的精度使我们能够将速度估计到大约1 m/s左右，我将降低这种精度以说明传感器融合的效果。让我们将范围误差改为$\sigma=500$米，然后计算估计速度的标准差。我会跳过前几个测量，因为滤波器在那段时间内正在收敛，导致人为的大偏差。

不使用多普勒的标准差为：

In [ ]:
range_std = 500.
elevation_angle_std = math.degrees(0.5)
np.random.seed(200)
pos = (0, 0)
radar = RadarStation(pos, range_std, elevation_angle_std)
ac = ACSim(ac_pos, (100, 0), 0.02)

kf_sf = cv_UKF(f_cv_radar, h_radar, R_std=[range_std, elevation_angle_std])
time = np.arange(0, 360 + dt, dt)
xs = []
for _ in time:
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    kf_sf.predict()
    kf_sf.update([r[0], r[1], ac.vel[0], ac.vel[1]]) 
    xs.append(kf_sf.x)

xs = np.asarray(xs)
plot_radar(xs, time, plot_x=False, plot_vel=True, plot_alt=False)
print(f'速度标准差 {np.std(xs[10:, 1]):.1f} m/s')

对于多普勒，我们需要将 $x$ 和 $y$ 方向上的速度包含在测量中。`ACSim` 类将速度存储在数据成员 `vel` 中。为了执行卡尔曼滤波更新，我们只需要使用包含斜距离、仰角以及 $x$ 和 $y$ 方向上的速度的列表调用 `update`：

$$z = [\mathtt{slant\_range},\, \text{elevation angle},\, \dot x,\, \dot y]$$

测量包含四个值，因此测量函数还需要返回四个值。倾斜距离和仰角将像以前一样计算，我们不需要计算 $x$ 和 $y$ 的速度，因为它们由状态估计提供。

In [ ]:
def h_vel(x):
    dx = x[0] - h_vel.radar_pos[0]
    dz = x[2] - h_vel.radar_pos[1]
    slant_range = math.sqrt(dx**2 + dz**2)
    elevation_angle = math.atan2(dz, dx)
    return slant_range, elevation_angle, x[1], x[3]

现在我们可以实现我们的滤波器。

In [ ]:
h_radar.radar_pos = (0, 0)
h_vel.radar_pos = (0, 0)

range_std = 500.
elevation_angle_std = math.degrees(0.5)
vel_std = 2.

np.random.seed(200)
ac = ACSim(ac_pos, (100, 0), 0.02)
radar = RadarStation((0, 0), range_std, elevation_angle_std)

kf_sf2 = cv_UKF(f_cv_radar, h_vel, 
            R_std=[range_std, elevation_angle_std, vel_std, vel_std])

In [ ]:
time = np.arange(0, 360 + dt, dt)
xs = []
for t in time:
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    # 模拟多普勒速度读数
    vx = ac.vel[0] + randn()*vel_std
    vz = ac.vel[1] + randn()*vel_std
    kf_sf2.predict()
    kf_sf2.update([r[0], r[1], vx, vz]) 
    xs.append(kf_sf2.x)
xs = np.asarray(xs)
plot_radar(xs, time, plot_x=False, plot_vel=True, plot_alt=False)
print(f'速度标准差为 {np.std(xs[10:,1]):.1f} m/s')

通过加入速度传感器，我们成功将标准差从3.5米/秒降低到1.3米/秒。

传感器融合是一个大的话题，这只是一个相当简单的实现。在典型的导航问题中，我们有提供互补信息的传感器。例如，GPS可能每秒提供略微准确的位置更新，但速度估计很差，而惯性系统可能以50Hz提供非常准确的速度更新，但位置估计很差。每个传感器的优点和缺点是相互独立的。这导致了“互补滤波器” ，它将惯性高更新速度的速度测量值与GPS准确但更新速度缓慢的位置估计混合，产生高速和准确的位置和速度估计。高速度估计在GPS更新之间积分，以产生准确和高速的位置估计。

### 多个位置传感器

上一个传感器融合问题是一个玩具例子。让我们来解决一个不那么玩具化的问题。在 GPS 出现之前，船舶和飞机通过各种距离和方位系统（例如 VOR、LORAN、TACAN、DME 等）进行导航。这些系统以无线电波的形式发射信标。传感器从信号中提取到信标的距离和/或方位。例如，一架飞机可能有两个 VOR 接收器。飞行员将每个接收器调谐到不同的 VOR 站点。每个 VOR 接收器都显示*射线*——从地面的 VOR 站点到飞机的方向。飞行员使用图表找到射线的交点，确定飞机的位置。

这是一种精度较低的手动方法。使用卡尔曼滤波器将产生更准确的位置估计。假设我们有两个传感器，每个传感器仅提供到目标的方位角测量值，如下图所示。周围的边界宽度与传感器噪声的$3\sigma$成比例。飞行器必须位于两个边界的交点处，且有很高的概率性。

In [ ]:
ukf_internal.show_two_sensor_bearing()

我们根据传感器和目标之间的方位角计算方位角：

In [ ]:
def bearing(sensor, target_pos):
    return math.atan2(target_pos[1] - sensor[1], 
                      target_pos[0] - sensor[0])

滤波器以向量形式接收两个传感器的测量值。代码将接受任何可迭代容器，因此我使用Python列表以提高效率。我们可以实现为：

In [ ]:
def measurement(A_pos, B_pos, pos):
    angle_a = bearing(A_pos, pos) # 计算A站的角度
    angle_b = bearing(B_pos, pos) # 计算B站的角度
    return [angle_a, angle_b]


假设飞机的速度是恒定的。为了改变步调，我显式计算了新的位置，而不是使用矩阵-向量乘法：

In [ ]:
def fx_VOR(x, dt):
    x[0] += x[1] * dt # 更新x轴坐标
    x[2] += x[3] * dt # 更新y轴坐标
    return x

接下来实现测量函数。它将先前的结果转换为一个包含两个站点测量结果的数组。我不喜欢全局变量，但将站点的位置放在全局变量 `sa_pos` 和 `sb_pos` 中，以演示此共享数据的方法：

In [ ]:
sa_pos = [-400, 0]
sb_pos = [400, 0]

def hx_VOR(x):
    # 测量A站
    pos = (x[0], x[2])
    return measurement(sa_pos, sb_pos, pos)

现在我们编写样板文件来构建过滤器，运行它并绘制结果：

In [ ]:
def moving_target_filter(pos, std_noise, Q, dt=0.1, kappa=0.0):
    points = MerweScaledSigmaPoints(n=4, alpha=.1, beta=2., kappa=kappa)
    f = UKF(dim_x=4, dim_z=2, dt=dt, 
            hx=hx_VOR, fx=fx_VOR, points=points)
    f.x = np.array([pos[0], 1., pos[1], 1.])

    q = Q_discrete_white_noise(2, dt, Q)
    f.Q[0:2, 0:2] = q
    f.Q[2:4, 2:4] = q
    f.R *= std_noise**2
    f.P *= 1000    
    return f

def plot_straight_line_target(f, std_noise):
    xs, txs = [], []
    for i in range(300):
        target_pos[0] += 1 + randn()*0.0001
        target_pos[1] += 1 + randn()*0.0001
        txs.append((target_pos[0], target_pos[1]))

        z = measurement(sa_pos, sb_pos, target_pos)
        z[0] += randn() * std_noise
        z[1] += randn() * std_noise

        f.predict()
        f.update(z)
        xs.append(f.x)

    xs = np.asarray(xs)
    txs = np.asarray(txs)

    plt.plot(xs[:, 0], xs[:, 2])
    plt.plot(txs[:, 0], txs[:, 1], ls='--', lw=2, c='k')
    plt.show()

np.random.seed(123)
目标位置 = [100, 200]

std_noise = math.radians(0.5)
f = 移动目标滤波器(目标位置, std_noise, Q=1.0)
绘制直线目标(f, std_noise)

In [ ]:

我认为这看起来很不错。轨道的开始部分出现了较大的误差，但滤波器稳定下来并产生了良好的估计。

让我们重新审视角度的非线性。我将把目标位置放置在两个传感器之间的(0,0)点。这将导致residual的计算中出现非线性，因为平均角度将接近零。当角度低于0时，测量函数将计算出一个接近$2\pi$的大正角度。因此，预测和测量之间的residual将非常大，接近$2\pi$而不是接近0。这使得滤波器无法准确地执行，如下例所示。

```python
目标位置 = [0, 0]
f = 移动目标滤波器(目标位置, std_noise, Q=1.0)
绘制直线目标(f, std_noise)

这样的表现是不可接受的。`FilterPy`的UKF代码允许您指定一个函数，以在类似这种非线性情况下计算残差。本章的最后一个示例演示了它的用法。

## 传感器误差和几何效应

与被跟踪对象的传感器的几何形状有关的物理限制可能非常难以处理，当设计滤波器时。如果VOR站的径线几乎平行，则非常小的角度误差会转化为非常大的距离误差。更糟糕的是，这种行为是非线性的 - *x轴*与*y轴*的误差将根据实际方位角而变化。这些散点图显示了两个不同方位角的1°$\sigma$误差的误差分布。

In [ ]:
ukf_internal.plot_scatter_of_bearing_error()

## 练习：解释滤波器的性能

我们可以看到，对于小的角度误差，位置误差非常大。解释一下我们如何在上述目标跟踪问题中获得了相对良好的UKF性能。分别回答单个传感器和多个传感器的问题。

### 解决方案

这非常重要，需要努力在查看以下答案之前回答此问题。如果您无法回答此问题，则可能需要重新访问“多维Kalman滤波器”一章中的某些早期材料。

有几个因素导致了我们的成功。首先，让我们考虑只有一个传感器的情况。任何单个测量都具有极端的可能位置范围。但是，我们的目标正在移动，UKF正在考虑这一点。让我们绘制连续进行的几次测量的结果，以跟踪移动目标。

In [ ]:
ukf_internal.plot_scatter_moving_target()

每个单独的测量都有很大的位置误差。然而，连续测量的图表显示出明显的趋势——目标显然朝着右上方移动。当Kalman滤波器计算Kalman增益时，它通过使用测量函数考虑误差分布。在这个例子中，误差位于大约45°的线上，因此滤波器将折扣这个方向的误差。另一方面，与此正交的测量几乎没有误差，同样Kalman增益也会考虑到这一点。

这张图使得看起来很容易，因为我们为每个位置更新绘制了100个测量。飞机的移动是显而易见的。相比之下，Kalman滤波器每次更新只得到一个测量。因此，滤波器将无法产生与点状绿线所示一样好的拟合。

现在考虑一下，方位角没有提供任何距离信息。假设我们将初始估计值设为距传感器1000公里（实际距离为7.07公里），并使 $\mathbf P$ 非常小。在那个距离下，1° 的误差会导致位置误差达到17.5公里。由于滤波器对其位置估计非常确定且测量中没有提供距离信息，因此 KF 永远无法收敛到实际目标位置。

现在让我们考虑添加第二个传感器的影响。以下是两个图，显示不同传感器放置的效果。我使用正方形和三角形作为两个传感器的符号，并使用相同的符号形状和颜色绘制由每个传感器引起的误差分布。然后计算与两个带噪声的方位角测量对应的 $(x, y)$ 坐标，并用红点绘制它们，以显示 $x$ 和 $y$ 中噪声测量的分布。

In [ ]:
with figsize(10,5):
    ukf_internal.plot_iscts_two_sensors()

在第一个图中，我将传感器放置在目标的初始位置几乎正交的位置，因此我们得到了这些可爱的'x'形交叉点。我们可以看到随着目标移动，$x$ 和 $y$ 的误差如何改变，红色散点的形状随之变化-随着目标远离传感器但更接近传感器 B 的 $y$ 坐标，形状变得强烈椭圆。

在第二个图中，飞机从一个传感器附近起飞，然后飞过第二个传感器。误差的交点非常非正交，导致位置误差变得非常分散。

## UKF 的实现

FilterPy 实现了 UKF，但学习如何将方程转换为代码非常有启发性。实现 UKF 相当简单。首先，让我们编写计算 sigma 点的均值和协方差的代码。

我们将在矩阵中存储 sigma 点和权重，如下所示：

$$ 
\begin{aligned}
\text{权重} &= 
\begin{bmatrix}
w_0& w_1 & \dots & w_{2n}
\end{bmatrix} 
\\
\text{西格玛} &= 
\begin{bmatrix}
\mathcal{X}_{0,0} & \mathcal{X}_{0,1} & \dots & \mathcal{X}_{0,n-1} \\
\mathcal{X}_{1,0} & \mathcal{X}_{1,1} &  \dots & \mathcal{X}_{1,n-1} \\
\vdots & \vdots &  \ddots & \vdots \\
\mathcal{X}_{2n,0} & \mathcal{X}_{2n,1} & \dots & \mathcal{X}_{2n,n-1}
\end{bmatrix}
\end{aligned}
$$


有很多下标来描述一个非常简单的东西，所以这里有一个二维问题($n$=2)的例子：

In [ ]:
points = MerweScaledSigmaPoints(n=2, alpha=.1, beta=2., kappa=1.)
points.sigma_points(x=[0.,0], P=[[1.,.1],[.1, 1]])

平均值的西格玛点在第一行。它的位置是(0, 0)，等于平均值(0,0)。第二个西格玛点的位置是(0.173, 0.017)，依此类推。总共有$2n+1 = 5$行，每个西格玛点一行。如果$n=3$，那么会有$3$列和$7$行。

存储sigma值的行列顺序与列行顺序之间的选择有些随意。我的选择使得代码的其余部分更清晰，因为我可以将第i个sigma点称为“sigmas[i]”，而不是“sigmas [:,i]”。

### 权重

使用NumPy计算权重非常简单。回想一下Van der Merwe缩放的sigma点实现状态：

$$
\begin{aligned}
\lambda&=\alpha^2(n+\kappa)-n \\ 
W^m_0 &= \frac{\lambda}{n+\lambda} \\
W^c_0 &= \frac{\lambda}{n+\lambda} + 1 -\alpha^2 + \beta \\
W^m_i = W^c_i &= \frac{1}{2(n+\lambda)}\;\;\;i=1..2n
\end{aligned}
$$
    
这些的代码如下：

In [ ]:
lambda_ = alpha**2 * (n + kappa) - n
Wc = np.full(2*n + 1,  1. / (2*(n + lambda_))
Wm = np.full(2*n + 1,  1. / (2*(n + lambda_))
Wc[0] = lambda_ / (n + lambda_) + (1. - alpha**2 + beta)
Wm[0] = lambda_ / (n + lambda_)

我在“lambda_”中使用下划线，因为“lambda”是Python中的保留字。带有下划线的尾部是Pythonic的解决方法。

### Sigma点

sigma点的方程式为：

$$
\begin{cases}
\mathcal{X}_0 = \mu \\
\mathcal{X}_i = \mu +  \left[\sqrt{(n+\lambda)\Sigma} \right]_i, & i=1..n \\
\mathcal{X}_i = \mu - \left[\sqrt{(n+\lambda)\Sigma}\right]_{i-n} & i=(n+1)..2n
\end{cases}
$$

一旦我们理解了$\left[\sqrt{(n+\lambda)\Sigma}  \right]_i$这个术语，Python就不难。

术语$\sqrt{(n+\lambda)\Sigma}$是一个矩阵，因为$\Sigma$是一个矩阵。在$[\sqrt{(n+\lambda)\Sigma}]_i$中的下标$i$是选择矩阵的第i行向量。矩阵的平方根是什么？没有唯一的定义。一个定义是，矩阵$\Sigma$的平方根是矩阵$S$，当其与自身相乘时，会得到$\Sigma$：如果$\Sigma = SS$，则$S = \sqrt{\Sigma}$。

我们将选择另一种定义，具有使其更易于计算的数值特性。我们可以将平方根定义为矩阵S，当其与其转置相乘时，返回$\Sigma$：

$$
\Sigma = \mathbf{SS}^\mathsf T
$$

这个定义很受欢迎，因为 $\mathbf S$ 是使用 [*Cholesky decomposition*](https://en.wikipedia.org/wiki/Cholesky_decomposition) [5] 计算的。它将一个海尔米特正定矩阵分解为一个三角矩阵和它的共轭转置。该矩阵可以是上三角形或下三角形，如下所示：

$$A=LL^{∗} \\ A=U^{∗}U$$

星号表示共轭转置；因为我们只有实数，所以我们可以写成：

$$A=LL^\mathsf T \\ A=U^\mathsf T U$$

$\mathbf P$ 具有这些属性，因此我们可以将 $\mathbf S = \text{cholesky}(\mathbf P)$ 视为 $\mathbf P$ 的平方根。

SciPy 在 `scipy.linalg` 中提供了 `cholesky()` 方法。如果您选择的编程语言是 Fortran、C 或 C++，则 LAPACK 等库也提供此例程。Matlab 提供了 `chol()`。

默认情况下，`scipy.linalg.cholesky()`返回一个上三角矩阵，因此我编写代码以期望一个上三角矩阵。因此，我通过行访问结果，这样第一个sigma点（中心点）受到一整行非零值的影响。如果您提供自己的平方根实现，您需要考虑到这一点。您会在文献中找到以列优先方式获取值的UKF算法。如果cholesky是下三角形的，或者您正在使用计算对称矩阵的不同算法，因此行与列的顺序无关紧要，则可以使用此方法。

In [ ]:
import scipy
a = np.array([[2., .1], [.1, 3]])
s = scipy.linalg.cholesky(a)
print("cholesky:")
print(s)
print("\nsquare of cholesky:")
print(s @ s.T)

因此，我们可以使用以下代码实现sigma点：

In [ ]:
sigmas = np.zeros((2*n+1, n))
U = scipy.linalg.cholesky((n+lambda_)*P) # 平方根

In [ ]:
sigmas[0] = X
for k in range (n):
    sigmas[k+1]   = X + U[k]
    sigmas[n+k+1] = X - U[k]

现在让我们实现无味变换。回想一下方程式

$$\begin{aligned}
\mu &= \sum_i w_i^m\mathcal{X}_i \\
\Sigma &= \sum_i w_i^c{(\mathcal{X}_i-\mu)(\mathcal{X}_i-\mu)^\mathsf{T}}
\end{aligned}
$$

我们用以下代码实现均值之和

In [ ]:
x = np.dot(Wm, sigmas)

如果你不是 NumPy 的重度用户，这可能看起来很陌生。NumPy 不仅仅是一个使线性代数成为可能的库；在内部，它是用 C 和 Fortran 编写的，以实现比 Python 更快的速度。典型的加速比是 20 倍到 100 倍。为了获得这种加速，我们必须避免使用 for 循环，而是使用 NumPy 的内置函数来执行计算。因此，我们不是编写 for 循环来计算积的总和，而是调用内置的 `numpy.dot(x, y)` 方法。两个向量的点积是每个元素的逐元素乘法的总和。如果传递了一个 1D 数组和一个 2D 数组，它将计算内积的总和：

In [ ]:
a = np.array([10, 100])
b = np.array([[1, 2, 3],
              [4, 5, 6]])
np.dot(a, b)

现在只需要计算 $\mathbf P = \sum_i w_i{(\mathcal{X}_i-\mu)(\mathcal{X}_i-\mu)^\mathsf{T}} + \mathbf Q$：

In [ ]:
kmax, n = sigmas.shape
P = zeros((n, n))
for k in range(kmax):
    y = sigmas[k] - x
    P += Wc[k] * np.outer(y, y) 
P += Q

这里我们介绍NumPy的另一个特性。状态变量`x`是一维的，`sigmas[k]`也是一维的，所以差值`sigmas[k]-X`也是一维的。NumPy不会计算一个1-D数组的转置；它认为`[1,2,3]`的转置是`[1,2,3]`。因此，我们调用函数`np.outer(y,y)`计算1D数组$\mathbf{y}$的$\mathbf{yy}^\mathsf{T}$值。另一种实现方式可能是：

In [ ]:
y = (sigmas[k] - x).reshape(kmax, 1) # 转换成2D数组
P += Wc[K] * np.dot(y, y.T)

这段代码速度较慢，不够惯用，因此我们不会使用它。

### 预测步骤

对于预测步骤，我们将按照上述规定生成权重和sigma点。我们将每个sigma点通过函数f传递。

$$\boldsymbol{\mathcal{Y}} = f(\boldsymbol{\chi})$$

然后我们使用无损变换计算预测的均值和协方差。在下面的代码中，您可以看到我假设这是存储滤波器所需的各种矩阵和向量的类中的方法。

In [ ]:
def predict(self, sigma_points_fn):
    """ 执行UKF的预测步骤。返回时，self.xp和self.Pp包含预测状态（xp）和协方差（Pp）。
    'p'代表预测。
    """

    # 计算给定均值和协方差的sigma点
    sigmas = sigma_points_fn(self.x, self.Pp)

    for i in range(self._num_sigmas):
        self.sigmas_f[i] = self.fx(sigmas[i], self._dt)

    self.xp, self.Pp = unscented_transform(
                       self.sigmas_f, self.Wm, self.Wc, self.Q)

### 更新步骤

更新步骤通过函数`h(x)`将sigma转换为测量空间。

$$\mathcal{Z} = h(\mathcal{Y})$$

使用无迹变换，计算这些点的均值和协方差。然后计算残差和卡尔曼增益。交叉方差计算如下：

$$\mathbf P_{xz} =\sum_{i=0}^{2n} w^c_i(\boldsymbol{\mathcal Y}_i-\mu)(\boldsymbol{\mathcal Z}_i-\mu_z)^\mathsf T$$

最后，使用残差和卡尔曼增益计算新的状态估计：

$$\begin{aligned}
K &= \mathbf P_{xz} \mathbf P_z^{-1}\\
{\mathbf x} &= \mathbf{\bar x} + \mathbf{Ky}
\end{aligned}$$

新的协方差计算如下：

$$ \mathbf P = \mathbf{\bar P} - \mathbf{KP}_z\mathbf{K}^\mathsf{T}$$

假设这是一个存储必要矩阵和数据的类的方法，可以按以下方式实现。

In [ ]:
def update(self, z):
    # 更名以提高可读性
    sigmas_f = self.sigmas_f
    sigmas_h = self.sigmas_h

    # 将 sigma 点转换为测量空间
    for i in range(self._num_sigmas):
        sigmas_h[i] = self.hx(sigmas_f[i])

# 通过UT传递的预测均值和协方差
    zp，Pz = unscented_transform（sigmas_h，self.Wm，self.Wc，self.R）

In [ ]:
# 计算状态和测量之间的交叉协方差
Pxz = np.zeros（（self._dim_x，self._dim_z））
for i in range（self._num_sigmas）：
    Pxz + = self.Wc [i] * np.outer（sigmas_f [i] - self.xp，
                                sigmas_h [i] - zp）

K = np.dot（Pxz，inv（Pz））＃卡尔曼增益

self.x = self.xp + np.dot（K，z - zp）
self.P = self.Pp - np.dot（K，Pz）点（K.T）

In [ ]:

###FilterPy的实现

FilterPy在某种程度上进行了通用化。您可以指定不同的sigma点算法，指定如何计算状态变量的residual（无法减去角度，因为它们是模块化的），提供矩阵平方根函数等。有关详细信息，请参见帮助。

https://filterpy.readthedocs.org/#unscented-kalman-filter

##批量处理

Kalman滤波器是递归的——估计值基于当前测量和先前估计。但是通常会有一组已经收集好的data需要进行滤波。在这种情况下，滤波器可以以批处理模式运行，在此模式下，所有测量值都会被同时过滤。

将您的测量收集到一个数组或列表中。

```python
zs = read_altitude_from_csv()

然后调用`batch_filter()`方法。

In [ ]:
Xs, Ps = ukf.batch_filter(zs)

该函数获取测量值的列表/数组，对其进行滤波，并返回整个数据集的状态估计（`Xs`）和协方差矩阵（`Ps`）的数组。

以下是一个完整的示例，从上面的雷达跟踪问题中绘制而来。

In [ ]:
dt = 12. # 读数之间的12秒间隔
range_std = 5 # 米
bearing_std = math.radians(0.5)

ac_pos = (0., 1000.)
ac_vel = (100., 0.)
radar_pos = (0., 0.)
h_radar.radar_pos = radar_pos

In [ ]:
points = MerweScaledSigmaPoints(n=3, alpha=.1, beta=2., kappa=0.)
kf = UKF(3, 2, dt, fx=f_radar, hx=h_radar, points=points)

kf.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=dt, var=0.1)
kf.Q[2, 2] = 0.1

kf.R = np.diag([range_std**2, bearing_std**2])
kf.x = np.array([0., 90., 1100.])
kf.P = np.diag([300**2, 30**2, 150**2])

radar = RadarStation((0, 0), range_std, bearing_std)
ac = ACSim(ac_pos, (100, 0), 0.02)

np.random.seed(200)

t = np.arange(0, 360 + dt, dt)
n = len(t)

zs = []
for i in range(len(t)):
    ac.update(dt)
    r = radar.noisy_reading(ac.pos)
    zs.append([r[0], r[1]])

xs, covs = kf.batch_filter(zs)
ukf_internal.plot_radar(xs, t)

## 平滑处理结果

假设我们正在跟踪一辆汽车。假设我们得到一个嘈杂的测量，表明汽车开始向左转，但状态函数预测汽车直行。卡尔曼滤波器别无选择，只能将状态估计值在一定程度上向嘈杂的测量移动，因为它无法判断这是否只是一个特别嘈杂的测量或者是一个真正的转弯开始。

如果我们正在收集数据并进行后处理，我们会在疑问测量之后得到一些测量，告诉我们是否进行了转弯。假设接下来的测量都继续向左转。我们可以确定，这个测量不是很嘈杂，而是开始了一个转弯。

我们不会在这里开发数学或算法，我只会向您展示如何在 `FilterPy` 中调用算法。我们实现的算法称为 *RTS平滑器*，以算法的三位发明者Rauch、Tung和Striebel命名。

例程是 `UnscentedKalmanFilter.rts_smoother()`。使用它非常简单；我们将从 `batch_filter` 步骤计算出的均值和协方差传递进去，然后收到平滑后的均值、协方差和卡尔曼增益。

In [ ]:
Ms, P, K = kf.rts_smoother(xs, covs)
ukf_internal.plot_rts_output(xs, Ms, t)

从这些图表中，我们可以看到位置的改善很小，但速度的改善很好，高度则非常显著。位置上的差异非常小，因此我打印了 UKF 和平滑结果之间最后5个数据点的差异。如果可以对数据进行后处理，我建议始终使用 RTS 平滑器。

## 选择 Sigma 参数

我发现选择 $\alpha$、$\beta$ 和 $\kappa$ 值的文献相当缺乏。Van der Merwe 的论文包含了最多的信息，但并不详尽。因此，让我们探讨一下他们的做法。

Van der Merwe建议在高斯问题中使用$\beta=2$，$\kappa=3-n$。所以让我们从那里开始，变化$\alpha$。我会让$n=1$，以最小化我们需要查看的数组大小，并避免计算矩阵的平方根。

In [ ]:
from kf_book.ukf_internal import print_sigmas
print_sigmas(mean=0, cov=3, alpha=1)

这里发生了什么？我们可以看到对于均值为0，算法选择了0、3和-3的sigma点，但为什么？回想一下计算sigma点的方程式：

$$\begin{aligned}
\mathcal{X}_0 &= \mu\\
\mathcal{X}_i &= \mu \pm \sqrt{(n+\lambda)\Sigma}
\end{aligned}$$

我选择$n=1$将所有内容缩减为标量，使我们避免计算矩阵的平方根。因此，对于我们的值，方程式为

$$\begin{aligned}
\mathcal{X}_0 &= 0 \\
\mathcal{X}_i &= 0 \pm \sqrt{(1+2)\times 3} \\
&= \pm 3
\end{aligned}$$

因此，随着$\alpha$的增大，sigma点会更加分散。让我们将其设置为荒谬的值。

In [ ]:
print_sigmas(mean=0, cov=3, alpha=200)

我们可以看到sigma点分布在100个标准偏差之上。如果我们的数据是高斯分布，那么我们将会把许多标准偏差之外的数据融合在一起；对于非线性问题，这不太可能产生好的结果。但是假设我们的分布不是高斯分布，而是有着非常肥的尾巴呢？我们可能需要从这些尾巴中采样以获得一个好的估计，因此增加 $\kappa$ （不是200，200太大了，使得sigma点的变化非常明显）是有意义的。

按照类似的思路，假设我们的分布几乎没有尾巴 - 概率分布看起来更像一个倒置的抛物线。在这种情况下，我们可能希望将sigma点拉得更靠近均值，以避免在永远不会有真实数据的区域进行采样。

现在让我们来看看权重的变化。当 $k+n=3$ 时，均值的权重为0.6667，两个离群 sigma 点的权重为0.1667。另一方面，当 $\alpha=200$ 时，均值权重飙升至0.99999，离群值的权重设置为0.000004。回想一下权重的方程：

$$\begin{aligned}
W_0 &= \frac{\lambda}{n+\lambda} \\
W_i &= \frac{1}{2(n+\lambda)}
\end{aligned}$$

我们可以看到，随着 $\lambda$ 的增大，均值的权重比例 ($\lambda/(n+\lambda)$) 接近于1，其余 sigma 点的权重比例趋近于0。这在协方差大小上不变。因此，随着我们从均值进行更远的采样，我们会给予这些样本更少的权重，如果我们非常接近均值进行采样，我们会给所有样本非常相似的权重。

然而，Van der Merwe 给出的建议是将 $\alpha$ 约束在 $0 \gt \alpha \ge 1$ 的范围内。他建议将 $\alpha$ 设为 $10^{-3}$。让我们尝试一下。

In [ ]:
print_sigmas(mean=0, cov=13, alpha=.001, kappa=0)

## 机器人定位 - 一个完整的实例

现在是时候解决一个重大的问题了。大多数书籍选择简单的、教科书式的问题，有简单的答案，你会想知道如何解决实际的问题。这个例子不会教你如何解决任何问题，但是它演示了在设计和实现滤波器时你将不得不考虑的类型的事情。

我们将考虑机器人定位的问题。在这种情况下，我们有一个机器人通过使用传感器检测地标在一个景观中移动。这可以是一个使用计算机视觉识别树木、建筑物和其他地标的自动驾驶汽车。它可能是一个吸尘你家的小型机器人，或者是一个仓库中的机器人。

机器人有4个轮子，采用汽车相同的配置。它通过旋转前轮来操纵。这会导致机器人在向前移动时围绕后轴旋转。这是非线性行为，我们需要对其进行建模。

机器人有一个传感器，可以给出它与景观中已知目标的大致距离和方位。这是非线性的，因为从距离和方位计算位置需要使用平方根和三角函数。

过程模型和测量模型都是非线性的。UKF 可以适应这两种模型，因此我们暂时得出结论，UKF 是解决此问题的可行选择。

### 机器人运动模型

首先近似地说，汽车在前进时通过旋转前轮来转向。汽车的前部沿着轮子指向的方向移动，同时围绕后轮旋转。由于摩擦力的影响、不同速度下橡胶轮胎的不同行为以及外侧轮胎需要行驶比内侧轮胎不同半径的道路，这个简单的描述变得复杂。精确地建模转向需要一组复杂的微分方程。

对于卡尔曼滤波，特别是对于低速机器人应用，已经发现了一种更简单的“自行车模型”，效果良好。以下是该模型的描述：

In [ ]:
ekf_internal.plot_bicycle()

在这里，我们可以看到前轮相对于轴距指向方向 $\alpha$。在短时间内，汽车向前移动，后轮向前移动并略微向内转动，如蓝色阴影轮胎所示。在这样一个短时间范围内，我们可以将其近似为以半径 $R$ 为转向的圆弧。我们可以用以下公式计算转向角 $\beta$：

$$\beta = \frac{d}{w} \tan{(\alpha)}$$

其中，给定前进速度 $v$ 时后轮行驶距离为 $d=v\Delta t$。转向半径 $R$ 如下所示：

$$R = \frac{d}{\beta}$$

在转弯开始之前，用 $\theta$ 表示机器人的方向，我们计算出位置 $C$ 为：

$$\begin{aligned}
C_x &= x - R\sin(\theta) \\
C_y &= y + R\cos(\theta)
\end{aligned}$$

经过时间 $\Delta t$ 后机器人的新位置和方向为：

$$\begin{aligned} \bar x &= C_x + R\sin(\theta + \beta) \\
\bar y &= C_y - R\cos(\theta + \beta) \\
\bar \theta &= \theta + \beta
\end{aligned}
$$

当我们替换 $C$ 后，可以得到：

$$\begin{aligned} \bar x &= x - R\sin(\theta) + R\sin(\theta + \beta) \\
\bar y &= y + R\cos(\theta) - R\cos(\theta + \beta) \\
\bar \theta &= \theta + \beta
\end{aligned}
$$

如果您对转向模型不感兴趣，就不需要详细理解这个公式。重要的是要认识到我们的运动模型是非线性的，因此我们需要使用卡尔曼滤波器来处理它。

### 设计状态变量

对于我们的机器人，我们将维护其位置和方向：

$$\mathbf x = \begin{bmatrix}x & y & \theta\end{bmatrix}^\mathsf{T}$$

我可以将速度包含在此模型中，但是您将会看到，数学运算已经相当具有挑战性。

控制输入 $\mathbf{u}$ 是指令速度和转向角度：

$$\mathbf{u} = \begin{bmatrix}v & \alpha\end{bmatrix}^\mathsf{T}$$

### 设计系统模型

我们将我们的系统建模为非线性运动模型加上白噪声。

$$\bar x = x + f(x, u) + \mathcal{N}(0, Q)$$

使用我们上面创建的机器人运动模型，我们可以编写如下代码：

In [ ]:
from math import tan, sin, cos, sqrt

def move(x, dt, u, wheelbase):
    hdg = x[2]
    vel = u[0]
    steering_angle = u[1]
    dist = vel * dt

    if abs(steering_angle) > 0.001: # 机器人是否在转弯？
        beta = (dist / wheelbase) * tan(steering_angle)
        r = wheelbase / tan(steering_angle) # 半径

        sinh, sinhb = sin(hdg), sin(hdg + beta)
        cosh, coshb = cos(hdg), cos(hdg + beta)
        return x + np.array([-r*sinh + r*sinhb, 
                              r*cosh - r*coshb, beta])
    else: # 直行
        return x + np.array([dist*cos(hdg), dist*sin(hdg), 0])

我们将使用此函数实现状态转移函数 `f(x)`。

我将设计 UKF，使得 $\Delta t$ 很小。如果机器人移动缓慢，那么这个函数应该会给出一个相当精确的预测。如果 $\Delta t$ 很大或者你的系统动力学非常非线性，这种方法将会失败。在这些情况下，你需要使用更复杂的数值积分技术（如 Runge-Kutta）来实现它。数值积分在 **Kalman 滤波器数学** 章节中有简要介绍。

### 设计测量模型

传感器提供了到景观中多个已知位置的噪声方位和距离。测量模型必须将状态 $\begin{bmatrix}x & y&\theta\end{bmatrix}^\mathsf{T}$ 转换为到地标的距离和方位角。如果 $p$ 是地标的位置，则距离 $r$ 为

$$r = \sqrt{(p_x - x)^2 + (p_y - y)^2}$$

我们假设传感器提供的是相对于机器人方向的方位角，因此我们必须从方位角中减去机器人的方向，以获得传感器读数，如下所示：

$$\phi = \tan^{-1}(\frac{p_y - y}{p_x - x}) - \theta$$

因此，我们的测量函数为

$$\begin{aligned}
\mathbf{z}& = h(\mathbf x, \mathbf P) &+ \mathcal{N}(0, R)\\
&= \begin{bmatrix}
\sqrt{(p_x - x)^2 + (p_y - y)^2} \\
\tan^{-1}(\frac{p_y - y}{p_x - x}) - \theta 
\end{bmatrix} &+ \mathcal{N}(0, R)
\end{aligned}$$

由于在*实现*一节中会讨论到的困难，我暂不会立即实现。

### 设计测量噪声

假设距离和方位角测量噪声是相互独立的，因此

$$\mathbf R=\begin{bmatrix}\sigma_{range}^2 & 0 \\ 0 & \sigma_{bearing}^2\end{bmatrix}$$

### 实现

在开始编码之前，我们还有一个问题要处理。残差为 $y = z - h(x)$。假设 $z$ 的方位角为 $1^\circ$，$h(x)$ 为 $359^\circ$。它们的差为 $-358^\circ$。这会影响卡尔曼增益的计算，因为正确的角度差为 $2^\circ$。因此，我们需要编写代码来正确计算方位角残差。

In [ ]:
def normalize_angle(x):
    x = x % (2 * np.pi)    # 强制在 [0, 2 pi) 范围内
    if x > np.pi:          # 转到 [-pi, pi) 范围内
        x -= 2 * np.pi
    return x

In [ ]:
print(np.degrees(normalize_angle(np.radians(1-359))))

状态向量在索引2处具有方位角，但测量向量在索引1处具有方位角，因此我们需要编写处理每个向量的函数。我们面临的另一个问题是，随着机器人的操纵，不同的地标将可见，因此我们需要处理可变数量的测量。测量中残差的函数将传递一个包含多个测量的数组，每个地标一个。

In [ ]:
def residual_h(a, b):
    y = a - b
    # 格式为 [dist_1, bearing_1, dist_2, bearing_2,...] 的数据
    for i in range(0, len(y), 2):
        y[i + 1] = normalize_angle(y[i + 1])
    return y

def residual_x(a, b):
    y = a - b
    y[2] = normalize_angle(y[2])
    return y

现在我们可以实现测量模型了。方程为
$$h(\mathbf x, \mathbf P)
= \begin{bmatrix}
\sqrt{(p_x - x)^2 + (p_y - y)^2} \\
\tan^{-1}(\frac{p_y - y}{p_x - x}) - \theta 
\end{bmatrix}$$

表达式$\tan^{-1}(\frac{p_y - y}{p_x - x}) - \theta$可能会产生$[-\pi, \pi)$之外的结果，因此我们应该将角度归一化到此范围内。

该函数将接收到一组地标，并需要生成一个形如`[dist_to_1, bearing_to_1, dist_to_2, bearing_to_2, ...]`的测量数组。

In [ ]:
def Hx(x, landmarks):
    """接收状态变量并返回与该状态对应的测量值。"""
    hx = []
    for lmark in landmarks:
        px, py = lmark
        dist = sqrt((px - x[0])**2 + (py - x[1])**2)
        angle = atan2(py - x[1], px - x[0])
        hx.extend([dist, normalize_angle(angle - x[2])])
    return np.array(hx)

我们的困难还没有结束。无香变换计算状态和测量矢量的平均值，但每个矢量都包含一个方位角。计算一组角度的平均值没有唯一的方法。例如，$359^\circ$和$3^\circ$的平均值是多少？直觉表明答案应该是$1^\circ$，但是一个简单的 $\frac{1}{n}\sum x$ 方法会得到$181^\circ$。

一种常见的方法是对正弦和余弦之和取反正切。

$$\bar{\theta} = atan2\left(\frac{\sum_{i=1}^n \sin\theta_i}{n}, \frac{\sum_{i=1}^n \cos\theta_i}{n}\right)$$

`UnscentedKalmanFilter.__init__()`有一个参数`x_mean_fn`，用于计算状态的平均值，以及一个参数`z_mean_fn`，用于计算测量的平均值。我们将编写以下函数：

In [ ]:
def state_mean(sigmas, Wm):
    x = np.zeros(3)

In [ ]:
sum_sin = np.sum(np.dot(np.sin(sigmas[:, 2]), Wm))
sum_cos = np.sum(np.dot(np.cos(sigmas[:, 2]), Wm))
x[0] = np.sum(np.dot(sigmas[:, 0], Wm))
x[1] = np.sum(np.dot(sigmas[:, 1], Wm))
x[2] = atan2(sum_sin, sum_cos)
return x

def z_mean(sigmas, Wm):
    z_count = sigmas.shape[1]
    x = np.zeros(z_count)

    for z in range(0, z_count, 2):
        sum_sin = np.sum(np.dot(np.sin(sigmas[:, z+1]), Wm))
        sum_cos = np.sum(np.dot(np.cos(sigmas[:, z+1]), Wm))

        x[z] = np.sum(np.dot(sigmas[:,z], Wm))
        x[z+1] = atan2(sum_sin, sum_cos)
    return x

这些函数利用了NumPy的三角函数操作数组，并且`dot`执行逐元素乘法。NumPy是用C和Fortran实现的，所以`sum(dot(sin(x), w))`比用Python编写等效的循环要快得多。

完成了这些，我们现在可以开始实现UKF了。我想指出的是，当我设计这个滤波器时，并不是一次性从头开始设计上面的所有函数。我先组装了一个基本的UKF，使用预定义的地标进行验证，然后开始填补空缺。“如果我看到不同的地标怎么办？”这让我改变了测量函数，使其接受一个地标数组。“如何计算角度的残差？”这让我编写了角度归一化代码。“一组角度的*平均值*是什么？”我在互联网上搜索，找到了维基百科上的一篇文章，并实现了该算法。不要感到畏惧。先设计能够设计的部分，然后一个一个地提出问题并解决。

你已经看到了UKF的实现，所以我不会详细描述它。这里有两件新事。当我们构造sigma点和滤波器时，我们必须提供我们编写的用于计算残差和平均值的函数。

In [ ]:
points = SigmaPoints(n=3, alpha=.00001, beta=2, kappa=0, 
                     subtract=residual_x)

ukf = UKF(dim_x=3, dim_z=2, fx=fx, hx=Hx, dt=dt, points=points,
         x_mean_fn=state_mean, z_mean_fn=z_mean,
         residual_x=residual_x, residual_z=residual_h)

接下来，我们需要将额外数据传递给我们的 `f(x, dt)` 和 `h(x)` 函数。我们想要使用 `move(x, dt, u, wheelbase)` 作为 `f(x, dt)`，并使用 `Hx(x, landmarks)` 作为 `h(x)`。我们可以这样做，只需要将额外的参数作为关键字参数传递到 `predict()` 和 `update()` 中，如下所示：

In [ ]:
            ukf.predict(u=u, wheelbase=wheelbase)        
            ukf.update(z, landmarks=landmarks)

代码的其余部分运行了仿真并绘制结果。我创建了一个变量`landmarks`，其中包含地标的坐标。我每秒更新模拟机器人的位置10次，但每秒只运行一次UKF。我们没有使用Runge-Kutta来积分运动的微分方程，因此小的时间步长使仿真更准确。

In [ ]:
from filterpy.stats import plot_covariance_ellipse

dt = 1.0
wheelbase = 0.5

def run_localization(
    cmds, landmarks, sigma_vel, sigma_steer, sigma_range, 
    sigma_bearing, ellipse_step=1, step=10):

    plt.figure()
    points = MerweScaledSigmaPoints(n=3, alpha=.00001, beta=2, kappa=0, 
                                    subtract=residual_x)
    ukf = UKF(dim_x=3, dim_z=2*len(landmarks), fx=move, hx=Hx,
              dt=dt, points=points, x_mean_fn=state_mean, 
              z_mean_fn=z_mean, residual_x=residual_x, 
              residual_z=residual_h)

In [ ]:
ukf.x = np.array([2, 6, .3])
ukf.P = np.diag([.1, .1, .05])
ukf.R = np.diag([sigma_range**2, 
                 sigma_bearing**2]*len(landmarks))
ukf.Q = np.eye(3)*0.0001

sim_pos = ukf.x.copy()

# 绘制地标
if len(landmarks) > 0:
    plt.scatter(landmarks[:, 0], landmarks[:, 1], 
                marker='s', s=60)

track = []
for i, u in enumerate(cmds):     
    sim_pos = move(sim_pos, dt/step, u, wheelbase)
    track.append(sim_pos)

    if i % step == 0:
        ukf.predict(u=u, wheelbase=wheelbase)

        if i % ellipse_step == 0:
            plot_covariance_ellipse(
                (ukf.x[0], ukf.x[1]), ukf.P[0:2, 0:2], std=6,
                 facecolor='k', alpha=0.3)

In [ ]:
x, y = sim_pos[0], sim_pos[1]
            z = []
            for lmark in landmarks:
                dx, dy = lmark[0] - x, lmark[1] - y
                d = sqrt(dx**2 + dy**2) + randn()*sigma_range
                bearing = atan2(lmark[1] - y, lmark[0] - x)
                a = (normalize_angle(bearing - sim_pos[2] + 
                     randn()*sigma_bearing))
                z.extend([d, a])            
            ukf.update(z, landmarks=landmarks)

            if i % ellipse_step == 0:
                plot_covariance_ellipse(
                    (ukf.x[0], ukf.x[1]), ukf.P[0:2, 0:2], std=6,
                     facecolor='g', alpha=0.8)
    track = np.array(track)
    plt.plot(track[:, 0], track[:,1], color='k', lw=2)
    plt.axis('equal')
    plt.title("无迹KalmanFilter机器人定位")
    plt.show()
    return ukf

In [ ]:
landmarks = np.array([[5, 10], [10, 5], [15, 15]])
cmds = [np.array([1.1, .01])] * 200
ukf = run_localization(
    cmds, landmarks, sigma_vel=0.1, sigma_steer=np.radians(1),
    sigma_range=0.3, sigma_bearing=0.1)
print('最终 P:', ukf.P.diagonal())

代码的剩余部分运行模拟并绘制结果。我创建了一个变量 `landmarks`，其中包含地标的坐标。我每秒更新模拟机器人的位置 10 次，但只运行 UKF 一次。这有两个原因。首先，我们没有使用 Runge Kutta 积分运动微分方程，因此较窄的时间步长使得模拟更准确。其次，在嵌入式系统中仅允许有限的处理速度是相当普遍的。这迫使您只在绝对必要的情况下运行卡尔曼滤波器。 

### 操控机器人

以上运行中的转向模拟不够逼真。速度和转向角度从未改变，这对于卡尔曼滤波器并不构成太大的问题。我们可以实现一个复杂的PID控制机器人模拟，但我将使用NumPy的`linspace`方法生成不同的转向指令。由于机器人将行驶的距离更远，我还会添加更多的地标。

In [ ]:
landmarks = np.array([[5, 10], [10, 5], [15, 15], [20, 5],
                      [0, 30], [50, 30], [40, 10]])
dt = 0.1
wheelbase = 0.5
sigma_range=0.3
sigma_bearing=0.1

def turn(v, t0, t1, steps):
  return [[v, a] for a in np.linspace(
                 np.radians(t0), np.radians(t1), steps)]  
    
# 从停止状态加速
cmds = [[v, .0] for v in np.linspace(0.001, 1.1, 30)]
cmds.extend([cmds[-1]]*50)

# 左转
v = cmds[-1][0]
cmds.extend(turn(v, 0, 2, 15))
cmds.extend([cmds[-1]]*100)

#右转
cmds.extend(turn(v, 2, -2, 15))
cmds.extend([cmds[-1]]*200)

In [ ]:
cmds.extend(turn(v, -2, 0, 15))
cmds.extend([cmds[-1]]*150)

cmds.extend(turn(v, 0, 1, 25))
cmds.extend([cmds[-1]]*100)

In [ ]:
ukf = run_localization(
    cmds, landmarks, sigma_vel=0.1, sigma_steer=np.radians(1),
    sigma_range=0.3, sigma_bearing=0.1, step=1,
    ellipse_step=20)
print('final covariance', ukf.P.diagonal())

不确定性很快就变得非常小。协方差椭圆显示的是$6\sigma$协方差，但椭圆非常小，很难看到。我们可以通过仅提供靠近起点的两个地标来将更多的误差纳入答案。当我们运行此滤波器时，随着机器人远离这些地标，误差增加。

In [ ]:
ukf = run_localization(
    cmds, landmarks[0:2], sigma_vel=0.1, sigma_steer=np.radians(1),
    sigma_range=0.3, sigma_bearing=0.1, step=1,
    ellipse_step=20)
print('final covariance', ukf.P.diagonal())

## 讨论

你对本章的印象可能取决于你以前实现过多少非线性卡尔曼滤波器。如果这是你第一次接触，也许计算 $2n+1$ 个 sigma 点并随后编写 $f(x)$ 和 $h(x)$ 函数会让你感到有些棘手。的确，由于需要处理角度的模数运算，我花了比我愿意承认的更多时间来使一切正常工作。另一方面，如果你已经实现了扩展卡尔曼滤波器 (EKF)，也许你正在座位上愉快地弹跳。编写 UKF 函数有一些繁琐，但概念非常基础。对于同样的问题，EKF 需要一些相当困难的数学。对于许多问题，我们无法找到 EKF 方程的闭式解，因此必须退回到某种迭代解决方案。

UKF相对于EKF的优势不仅在于相对容易的实现。讨论这个问题还为时过早，因为您还没有学习EKF，但是EKF在一个点上线性化问题，然后通过线性卡尔曼滤波器传递该点。相比之下，UKF采用$2n+1$个样本。因此，当问题高度非线性时，UKF通常比EKF更准确。虽然UKF并不保证总是优于EKF，但在实践中已经被证明至少表现得和EKF一样好，通常比EKF好得多。

因此，我的建议是始终从实施UKF开始。如果您的滤波器出现发散造成现实世界的后果（人员伤亡、大量资金损失、电站爆炸等），当然您将不得不进行复杂的分析和实验以选择最佳滤波器。这超出了本书的范围，您应该去研究生院学习这个理论。

最后，我提到UKF作为执行sigma点滤波器的“唯一”方法。这是不正确的。我选择的具体版本是Julier的缩放无味滤波器，由Van der Merwe在他的2004年论文中参数化。如果你搜索Julier，Van der Merwe，Uhlmann和Wan，你会发现他们开发的一系列类似的sigma点滤波器家族。每种技术都使用不同的方法选择和加权sigma点。但选择不止于此。例如，SVD Kalman滤波器使用奇异值分解（SVD）来找到概率分布的近似均值和协方差。把这一章看作是sigma点滤波器的介绍，而不是它们如何工作的明确处理。

## 参考文献

- [1] Rudolph Van der Merwe。“Sigma-Point Kalman Filters for Probabilistic Inference in Dynamic State-Space Models”dissertation（2004）。

- [2] Simon J. Julier. "The Scaled Unscented Transformation". 美国控制会议论文集6. IEEE. (2002)

- [3] http://www.esdradar.com/brochures/Compact%20Tracking%2037250X.pdf

- [4] Julier, Simon J.; Uhlmann, Jeffrey "卡尔曼滤波器对非线性系统的一种新扩展". Proc. SPIE 3068, 信号处理，传感器融合和目标识别 VI, 182 (1997 年 7 月 28 日)

- [5] Cholesky 分解。维基百科。http://en.wikipedia.org/wiki/Cholesky_decomposition